# 1 - Raster fields

The raster builders turn a pyramids `Dataset` into an interactive field. NoData is transparent
automatically, every styling keyword (`cmap`, `clim`, `levels`, ...) is validated against the element, and
each map carries a live colorbar, hover and zoom.

**Setup** and data - the German population grid and the global ETOPO relief.


In [ ]:
import warnings

warnings.filterwarnings("ignore")  # deliberately silence all warnings to keep the rendered docs clean

import io
from pathlib import Path

import holoviews as hv
import matplotlib.pyplot as plt
from IPython.display import Image

hv.extension("bokeh", "matplotlib")  # Bokeh = interactive (the default); matplotlib only for barbs / projection

# This notebook lives in docs/examples/interactive/guide/, so the bundled sample data at the repo-root
# examples/data/ is four levels up. The kernel's working directory is always the notebook's own folder
# (true for nbval, nbconvert and JupyterLab alike), so one clean relative path is all that is needed.
DATA = Path("../../../../examples/data")

from digitalearth.interactive import InteractiveMap
from pyramids.dataset import Dataset


def show(m, static=False):
    """Return the **live, interactive Bokeh map** - pan, zoom, hover, and the toolbar are all real.

    Run the notebook (Run All / Shift-Enter), or open it in a trusted viewer, to see the maps draw; a saved
    .ipynb only shows them once a kernel executes their JavaScript. A few layers have no Bokeh glyph
    (``projection``, ``streamlines``, wind ``barbs``) and are drawn by matplotlib; for those pass
    ``static=True`` (projected maps switch automatically) to embed a PNG instead.
    """
    if static or getattr(m, "_projection", None) is not None:
        fig = hv.render(m.render(), backend="matplotlib")
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
        plt.close(fig)
        return Image(data=buf.getvalue())
    return m.render()


In [ ]:
import numpy as np
from pyramids.dataset import GeoReference

pop = Dataset.read_file(str(DATA / "germany_pop_2020_1km.tif"))
relief = Dataset.read_file(str(DATA / "global" / "etopo_relief.tif"))
_v = pop.read_array(band=0)
cap = float(np.nanpercentile(_v[_v > 0], 99))
print("population:", pop.rows, "x", pop.columns, "| relief:", relief.rows, "x", relief.columns)

## `field` - a pixel grid

`field` draws the raster as a regular pixel grid. `cmap` names the colormap and `colorbar()` adds the
figure-level guide. A `clim` caps the colour range so the dense cities do not wash the map out.

In [ ]:
m = InteractiveMap(crs=pop.epsg, title="Germany population 2020 (persons / km2)")
m.field(pop, cmap="inferno", limits=(0.0, cap)).colorbar()
show(m)

## A frozen colour scale with `clim`

Passing `limits=(vmin, vmax)` pins the colour range instead of auto-ranging to the frame, so two rasters (or
two panels) share one scale and read comparably. `cnorm` picks the normalisation (`"linear"`, `"log"`,
`"eq_hist"`).

In [ ]:
m = InteractiveMap(crs=relief.epsg, title="Global relief - fixed -8000..8000 m scale")
m.field(relief, cmap="terrain", limits=(-8000.0, 8000.0)).colorbar()
show(m)

## `quadmesh` - irregular cells

`quadmesh` draws every cell from its coordinate arrays, so non-uniform or curvilinear grids render without
resampling. On a regular grid it matches `field`.

In [ ]:
m = InteractiveMap(crs=relief.epsg, title="Global relief (quadmesh)")
m.quadmesh(relief, cmap="viridis")
show(m)

## `contours` - iso-lines

`contours` draws iso-lines, filled or not, at a chosen number of `levels`. Layers chain, so a filled set
and a line set compose into one figure.

In [ ]:
m = InteractiveMap(crs=relief.epsg, title="Relief contours")
m.contours(relief, levels=12, filled=True).colorbar()
show(m)

## `rgb` - a three-band composite

`rgb` renders a three-band `Dataset` as a colour image. Here three months of global temperature (Jan / Jul /
Dec) become R / G / B, so seasonal change reads as colour.

In [ ]:
def _month(month):
    return Dataset.read_file(str(DATA / "global" / f"ncep_tavg_{month:02d}.tif")).read_array(band=0)

stack = np.stack([_month(1), _month(7), _month(12)]).astype("float32")
ref = Dataset.read_file(str(DATA / "global" / "ncep_tavg_01.tif"))
composite = Dataset.from_array(
    arr=stack, geo_ref=GeoReference(geo=ref.geotransform, epsg=ref.epsg), no_data_value=np.nan
)
m = InteractiveMap(crs=ref.epsg, title="Temperature Jan / Jul / Dec as R / G / B")
m.rgb(composite)
show(m)

## `large_image` - windowed reads for huge rasters

`large_image` reads only the pixels the viewport needs, re-reading on pan/zoom (a `RangeXY` stream) and on a
browser resize (a `PlotSize` stream), capped at `max_pixels` per frame. `dynamic=False` renders one
deterministic frame of the whole raster within the budget.

In [ ]:
m = InteractiveMap(crs=pop.epsg, title="population via large_image (budgeted)")
m.large_image(pop, dynamic=False, max_pixels=400 * 400, cmap="inferno")
show(m)